# Exploratory Data Analysis

This notebook examines the raw data sources, documents quality issues and validates the pipeline outputs.

In [ ]:
import polars as pl
from pathlib import Path
import json

Data Layer paths

In [ ]:
RAW = Path('../data/raw')
BRONZE = Path('../data/bronze')
SILVER = Path('../data/silver')
GOLD = Path('../data/gold')

### 1. Raw Data Exploration — CSV Files

In [ ]:
csv_files = sorted(RAW.glob('customers_*.csv'))

for data in csv_files:
   customer_df = pl.read_csv(data)
   print(f'  Rows: {customer_df.height}, Columns: {customer_df.width}')
   print(f'  Schema: {dict(zip(customer_df.columns, customer_df.dtypes))}')
   print(f'  Nulls: {customer_df.null_count().to_dicts()[0]}')
   print(customer_df.head(3))

### 2. Identify Data Quality issues

Name Casing Issues

In [ ]:
baseline_data = pl.read_csv(RAW / 'customers_1_full.csv')

print('Name casing examples:')
print(baseline_data.select('name').head(10))
print(f'\nAll uppercase: {baseline_data.filter(pl.col("name") == pl.col("name").str.to_uppercase()).height}')
print(f'All lowercase: {baseline_data.filter(pl.col("name") == pl.col("name").str.to_lowercase()).height}')
print(f'Mixed/Title:   {baseline_data.height - baseline_data.filter(pl.col("name") == pl.col("name").str.to_uppercase()).height - baseline_data.filter(pl.col("name") == pl.col("name").str.to_lowercase()).height}')

Duplicate customer_ids across incremental files

In [ ]:
all_dfs = [pl.read_csv(f) for f in csv_files]
combined_data = pl.concat(all_dfs)
print(f'Total rows across all CSV files: {combined_data.height}')
print(f'Unique customer_ids: {combined_data.select("customer_id").unique().height}')
dupes = combined_data.group_by('customer_id').agg(pl.count().alias('cnt')).filter(pl.col('cnt') > 1).sort('cnt', descending=True)
print(f'\nDuplicated customer_ids: {dupes.height}')
print(dupes.head(10))

### 3. JSON Products Structure

In [ ]:
with open(RAW / 'products.json') as f:
    products = json.load(f)

print(f'Number of products: {len(products)}')

for product in products:
    print(f'\n{product["product_id"]}: {product["name"]} ({product["category"]})')
    print(f'  Spec keys: {list(product["specs"].keys())}')
    print(f'  Base price: {product["pricing"]["base_price"]}')
    print(f'  Discounts: {len(product["pricing"]["discounts"])}')